In [1]:
import os
import sys
import json
import glob
import math
import shutil
import numpy as np
import pandas as pd
import cv2
from scipy import signal
from scipy.signal import periodogram
from tqdm import tqdm
import torch
import torch.multiprocessing
torch.multiprocessing.set_sharing_strategy('file_system')
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

REPO_ROOT = "/home/naver/disk2/HoangDPB/rPPG-Toolbox"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from neural_methods.model.iBVPNet import iBVPNet
from neural_methods.model.FactorizePhys.FactorizePhys import FactorizePhys

In [2]:
# ----- paths -----
RAW_DATA_PATH       = os.path.join(REPO_ROOT, "raw_data/reading")
PREPROCESSED_PATH   = os.path.join(REPO_ROOT, "preprocessed_data/groupF")
OUTPUT_DIR          = os.path.join(REPO_ROOT, "results/groupF/reading")

# ----- video / signal params -----
VIDEO_FPS   = 30       # camera frame rate
PPG_FS      = 25       # PPG sensor sampling rate (Hz)

# ----- preprocessing params -----
CHUNK_LENGTH = 160     # frames per clip
IMG_H, IMG_W = 72, 72  # face crop resolution
LABEL_TYPE   = "Standardized"  # z-score label (no cumsum in post-processing)

# ----- device -----
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

os.makedirs(PREPROCESSED_PATH, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("PREPROCESSED_PATH:", PREPROCESSED_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

Device: cuda:0
PREPROCESSED_PATH: /home/naver/disk2/HoangDPB/rPPG-Toolbox/preprocessed_data/groupF
OUTPUT_DIR: /home/naver/disk2/HoangDPB/rPPG-Toolbox/results/groupF/reading


In [3]:
# Model toggle

MODELS = [
    ("PURE_iBVPNet",              "iBVPNet",       "final_model_release/PURE_iBVPNet.pth"),
    ("PURE_FactorizePhys",        "FactorizePhys", "final_model_release/PURE_FactorizePhys_FSAM_Res.pth"),
    ("iBVP_FactorizePhys",      "FactorizePhys", "final_model_release/iBVP_FactorizePhys_FSAM_Res.pth"),
    ("SCAMPS_FactorizePhys",    "FactorizePhys", "final_model_release/SCAMPS_FactorizePhys_FSAM_Res.pth"),
    ("UBFC-rPPG_FactorizePhys", "FactorizePhys", "final_model_release/UBFC-rPPG_FactorizePhys_FSAM_Res.pth"),
]

print(f"Will run {len(MODELS)} model(s):")
for name, arch, path in MODELS:
    print(f"  {name}  ({arch})  ->  {path}")

Will run 5 model(s):
  PURE_iBVPNet  (iBVPNet)  ->  final_model_release/PURE_iBVPNet.pth
  PURE_FactorizePhys  (FactorizePhys)  ->  final_model_release/PURE_FactorizePhys_FSAM_Res.pth
  iBVP_FactorizePhys  (FactorizePhys)  ->  final_model_release/iBVP_FactorizePhys_FSAM_Res.pth
  SCAMPS_FactorizePhys  (FactorizePhys)  ->  final_model_release/SCAMPS_FactorizePhys_FSAM_Res.pth
  UBFC-rPPG_FactorizePhys  (FactorizePhys)  ->  final_model_release/UBFC-rPPG_FactorizePhys_FSAM_Res.pth


In [4]:
# Read video frames

def read_video_frames(video_path):
    """Read all frames from an MP4 file.

    Returns:
        frames (np.ndarray): shape (T, H, W, 3), dtype uint8, RGB order.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")

    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()

    if not frames:
        raise ValueError(f"Empty video: {video_path}")
    return np.stack(frames, axis=0)

In [5]:
# Read and time-align PPG signal

def read_ppg_synced(session_path, num_frames, fps=30):
    """Read PPG green channel from ppg.csv and resample to video frame times.

    The PPG sensor timestamps and video start timestamp are both in ms
    Unix epoch stored in metadata.json (sync_markers.video_start).
    Resampling is done by linear interpolation onto frame-spaced timestamps.

    Returns:
        ppg (np.ndarray): shape (num_frames,), dtype float32, raw green values.
    """
    meta_path = os.path.join(session_path, "metadata.json")
    with open(meta_path, "r") as f:
        meta = json.load(f)
    video_start_ms = meta["sync_markers"].get("video_start")
    if video_start_ms is None:
        video_start_ms = meta.get("start_timestamp")
    if video_start_ms is None:
        raise ValueError(f"No video_start or start_timestamp found in {meta_path}")
    video_start_ms = float(video_start_ms)

    ppg_df = pd.read_csv(os.path.join(session_path, "ppg.csv"))
    ppg_ts    = ppg_df["timestamp"].values.astype(np.float64)
    ppg_green = ppg_df["green"].values.astype(np.float64)

    # convert ppg timestamps to seconds relative to video start
    ppg_t = (ppg_ts - video_start_ms) / 1000.0

    # frame timestamps in seconds
    frame_t = np.arange(num_frames, dtype=np.float64) / fps

    # clip frame times to valid ppg range to avoid extrapolation
    frame_t_clipped = np.clip(frame_t, ppg_t[0], ppg_t[-1])

    ppg_resampled = np.interp(frame_t_clipped, ppg_t, ppg_green)
    return ppg_resampled.astype(np.float32)

In [6]:
# Normalization functions
# DATA_TYPE is Raw: no normalization on input frames (just face crop + resize)
# LABEL_TYPE is Standardized: z-score normalization on labels

def standardized_label(label):
    """Standardized label: global z-score."""
    label = label.astype(np.float64)
    m = np.mean(label)
    s = np.std(label)
    if s > 0:
        label = (label - m) / s
    else:
        label = np.zeros_like(label)
    return label.astype(np.float32)

In [7]:
# Face crop + resize

def crop_face_resize(frames, out_h, out_w, large_box_coef=1.5):
    """Detect face on frame 0, expand bbox by coef, resize all frames."""
    xml_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    detector  = cv2.CascadeClassifier(xml_path)

    frame0 = frames[0]
    if frame0.dtype != np.uint8:
        frame0 = np.clip(frame0, 0, 255).astype(np.uint8)
    gray = cv2.cvtColor(frame0, cv2.COLOR_RGB2GRAY)

    faces = detector.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)
    H, W = frames.shape[1], frames.shape[2]

    if len(faces) > 0:
        x, y, fw, fh = max(faces, key=lambda f: f[2])  # largest face
        x  = max(0, int(x  - (large_box_coef - 1.0) / 2.0 * fw))
        y  = max(0, int(y  - (large_box_coef - 1.0) / 2.0 * fh))
        fw = min(int(fw * large_box_coef), W - x)
        fh = min(int(fh * large_box_coef), H - y)
    else:
        x, y, fw, fh = 0, 0, W, H  # fallback: full frame

    C = frames.shape[3]
    resized = np.zeros((len(frames), out_h, out_w, C), dtype=np.float32)
    for i, frame in enumerate(frames):
        crop = frame[y : y + fh, x : x + fw]
        if crop.size == 0:
            crop = frame
        resized[i] = cv2.resize(crop.astype(np.float32), (out_w, out_h),
                                interpolation=cv2.INTER_AREA)
    return resized

In [8]:
# Discover subjects + read device-measured HR from hr.csv

def read_gt_hr(session_path):
    """Return average HR from hr.csv (hrStatus==1, hr>0).

    hr.csv has IBI columns whose values are arrays like [720, 695],
    which contain commas and break the standard CSV parser.
    Reading only the first 5 columns avoids the issue.
    """
    hr_path = os.path.join(session_path, "hr.csv")
    df = pd.read_csv(
        hr_path,
        usecols=[0, 1, 2, 3, 4],
        names=["id", "dataReceived", "timestamp", "hr", "hrStatus"],
        header=0,
    )
    valid = df[(df["hrStatus"] == 1) & (df["hr"] > 0)]["hr"]
    if len(valid) == 0:
        return float("nan")
    return float(valid.mean())


# Discover subject folders directly -- no metadata.csv required.
all_dirs = sorted([
    d for d in glob.glob(os.path.join(RAW_DATA_PATH, "*"))
    if os.path.isdir(d) and os.path.basename(d) != "videos"
])
print(f"Found {len(all_dirs)} subject folders\n")

subjects = []

for subj_dir in all_dirs:
    subj_id  = os.path.basename(subj_dir)
    subj_key = subj_id.replace("_", "")

    session_dirs = glob.glob(os.path.join(RAW_DATA_PATH, subj_id, "*"))
    if not session_dirs:
        print(f"No session folder for {subj_id}, skipping.")
        continue
    session_path = session_dirs[0]

    session_name  = os.path.basename(session_path)
    video_pattern = os.path.join(RAW_DATA_PATH, "videos",
                                  f"{subj_id}_{session_name}.mp4")
    video_files = glob.glob(video_pattern)
    if not video_files:
        print(f"No video found for {subj_id}, skipping.")
        continue
    video_path = video_files[0]

    gt_hr = read_gt_hr(session_path)

    subjects.append({
        "subj_id":      subj_id,
        "subj_key":     subj_key,
        "video_path":   video_path,
        "session_path": session_path,
        "gt_hr":        gt_hr,
    })
    print(f"  {subj_id}  HR_device={gt_hr:.4f} bpm  video={os.path.basename(video_path)}")

print(f"\nTotal subjects: {len(subjects)}")

Found 5 subject folders

  S_000  HR_device=80.4930 bpm  video=S_000_reading_1772099952592.mp4
  S_001  HR_device=87.9710 bpm  video=S_001_reading_1772100748921.mp4
  S_002  HR_device=74.0128 bpm  video=S_002_reading_1772101439887.mp4
  S_003  HR_device=65.4512 bpm  video=S_003_reading_1772102619267.mp4
  S_004  HR_device=76.8256 bpm  video=S_004_reading_1772103172990.mp4

Total subjects: 5


In [9]:
# Data preprocessing
# DATA_TYPE = Raw: face crop + resize only, NO normalization on input frames
# LABEL_TYPE = Standardized: z-score on labels
# Save as .npy with shape (CHUNK_LENGTH, H, W, 3)

# Clear any previous preprocessed data
if os.path.exists(PREPROCESSED_PATH):
    shutil.rmtree(PREPROCESSED_PATH)
os.makedirs(PREPROCESSED_PATH)
print(f"Cleared and recreated: {PREPROCESSED_PATH}\n")

all_input_files = []

for subj in subjects:
    subj_key     = subj["subj_key"]
    video_path   = subj["video_path"]
    session_path = subj["session_path"]

    print(f"=== Processing {subj_key} ===")

    frames = read_video_frames(video_path)
    T = frames.shape[0]
    print(f"  Video: {T} frames @ {VIDEO_FPS} fps")

    ppg_signal = read_ppg_synced(session_path, T, fps=VIDEO_FPS)
    print(f"  PPG green: min={ppg_signal.min():.0f}, max={ppg_signal.max():.0f}")

    # Face crop + resize (Raw data type: no normalization on frames)
    frames_cropped = crop_face_resize(frames, IMG_H, IMG_W)  # (T, 72, 72, 3) float32

    # Standardized label (z-score)
    label = standardized_label(ppg_signal)  # (T,)

    # Chunk into clips of CHUNK_LENGTH
    clip_num = T // CHUNK_LENGTH
    frame_clips = np.array([frames_cropped[i * CHUNK_LENGTH:(i + 1) * CHUNK_LENGTH] for i in range(clip_num)])
    label_clips = np.array([label[i * CHUNK_LENGTH:(i + 1) * CHUNK_LENGTH] for i in range(clip_num)])

    # Save per-subject subfolder
    subj_dir = os.path.join(PREPROCESSED_PATH, subj_key)
    os.makedirs(subj_dir)

    subj_files = []
    for chunk_idx in range(clip_num):
        input_path = os.path.join(subj_dir, f"{subj_key}_input{chunk_idx}.npy")
        label_path = os.path.join(subj_dir, f"{subj_key}_label{chunk_idx}.npy")

        np.save(input_path, frame_clips[chunk_idx])  # (CHUNK_LENGTH, H, W, 3)
        np.save(label_path, label_clips[chunk_idx])   # (CHUNK_LENGTH,)
        subj_files.append(input_path)

    all_input_files.extend(subj_files)
    print(f"  {clip_num} clips -> {subj_dir}\n")

print(f"Total clips saved: {len(all_input_files)}")
print("\nFolder structure:")
for subj in subjects:
    d = os.path.join(PREPROCESSED_PATH, subj["subj_key"])
    n = len(glob.glob(os.path.join(d, "*_input*.npy")))
    print(f"  {subj['subj_key']}/  ({n} clips)")

Cleared and recreated: /home/naver/disk2/HoangDPB/rPPG-Toolbox/preprocessed_data/groupF

=== Processing S000 ===
  Video: 2626 frames @ 30 fps
  PPG green: min=-71545, max=194479
  16 clips -> /home/naver/disk2/HoangDPB/rPPG-Toolbox/preprocessed_data/groupF/S000

=== Processing S001 ===
  Video: 2619 frames @ 30 fps
  PPG green: min=-227743, max=258441
  16 clips -> /home/naver/disk2/HoangDPB/rPPG-Toolbox/preprocessed_data/groupF/S001

=== Processing S002 ===
  Video: 2695 frames @ 30 fps
  PPG green: min=-187828, max=197372
  16 clips -> /home/naver/disk2/HoangDPB/rPPG-Toolbox/preprocessed_data/groupF/S002

=== Processing S003 ===
  Video: 2750 frames @ 30 fps
  PPG green: min=-401001, max=258429
  17 clips -> /home/naver/disk2/HoangDPB/rPPG-Toolbox/preprocessed_data/groupF/S003

=== Processing S004 ===
  Video: 2835 frames @ 30 fps
  PPG green: min=-17061, max=226073
  17 clips -> /home/naver/disk2/HoangDPB/rPPG-Toolbox/preprocessed_data/groupF/S004

Total clips saved: 82

Folder str

In [10]:
# PyTorch Dataset + DataLoader
# DATA_FORMAT: NCDHW -> transpose from (T, H, W, C) to (C, T, H, W)

class GroupFDataset(Dataset):

    def __init__(self, input_files):
        self.inputs = sorted(input_files)
        self.labels = [
            f.replace("input", "label")
            for f in self.inputs
        ]

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, index):
        data = np.float32(np.load(self.inputs[index]))   # (T, H, W, 3)
        label = np.float32(np.load(self.labels[index]))  # (T,)

        # NDHWC -> NCDHW: transpose (T, H, W, C) -> (C, T, H, W)
        data = np.transpose(data, (3, 0, 1, 2))  # (3, T, H, W)

        fname      = os.path.basename(self.inputs[index])
        split_idx  = fname.index("_")
        subject_id = fname[:split_idx]                     # e.g. "S000"
        chunk_id   = fname[split_idx + 6:].split(".")[0]  # +6 skips "_input"

        return data, label, subject_id, chunk_id


dataset = GroupFDataset(all_input_files)
print(f"Dataset: {len(dataset)} clips")

loader = DataLoader(dataset, batch_size=4, shuffle=False, num_workers=4)
print(f"DataLoader ready: {len(loader)} batches")

Dataset: 82 clips
DataLoader ready: 21 batches


In [11]:
# Post-processing functions
# LABEL_TYPE = Standardized -> diff_flag=False (NO cumsum)

def detrend(signal_in, lambda_val=100):
    """Smoothness-priors detrending (Tarvainen et al.)."""
    T_len = len(signal_in)
    H_mat = np.eye(T_len)
    ones  = np.ones(T_len)
    D_mat = (np.diag(ones[:-2], -2)
             - 2 * np.diag(ones[:-1], -1)
             + np.diag(ones))
    D_mat = D_mat[2:, :]
    inv   = np.linalg.inv(H_mat + lambda_val ** 2 * D_mat.T @ D_mat)
    return (H_mat - inv) @ signal_in


def bandpass_filter(sig, fs, low, high, order=1):
    """Zero-phase Butterworth bandpass filter."""
    b, a = signal.butter(order, [low / fs * 2, high / fs * 2], btype="bandpass")
    return signal.filtfilt(b, a, sig.astype(np.float64))


def fft_peak_hz(sig, fs, low, high):
    """Return dominant frequency (Hz) in [low, high] Hz via FFT."""
    N = 1
    while N < len(sig):
        N *= 2
    freqs, pxx = periodogram(sig, fs=fs, nfft=N, detrend=False)
    mask = (freqs >= low) & (freqs <= high)
    if not mask.any():
        return 0.0
    return float(freqs[mask][np.argmax(pxx[mask])])


def calculate_snr(pred_ppg, hr_label_bpm, fs, low_pass=0.6, high_pass=3.3):
    """Signal-to-noise ratio at HR harmonics vs background noise (dB)."""
    N = 1
    while N < len(pred_ppg):
        N *= 2
    freqs, pxx = periodogram(pred_ppg, fs=fs, nfft=N, detrend=False)

    f1  = hr_label_bpm / 60.0
    f2  = 2 * f1
    dev = 6.0 / 60.0  # +-6 bpm tolerance

    sig_mask   = (((freqs >= f1 - dev) & (freqs <= f1 + dev))
                  | ((freqs >= f2 - dev) & (freqs <= f2 + dev)))
    noise_mask = ((freqs >= low_pass) & (freqs <= high_pass) & ~sig_mask)

    sig_power   = pxx[sig_mask].sum()
    noise_power = pxx[noise_mask].sum()
    if noise_power == 0:
        return float("inf")
    return float(10.0 * np.log10(sig_power / noise_power))


def _reform_from_dict(chunk_dict):
    """Concatenate chunks in sorted key order into a 1-D array."""
    return np.concatenate([chunk_dict[k] for k in sorted(chunk_dict.keys())])


def process_bvp(pred_chunks, label_chunks, fs=30, diff_flag=False):
    """Post-process BVP predictions and labels.

    diff_flag=False for Standardized labels (no cumsum needed).
    """
    pred  = _reform_from_dict(pred_chunks).astype(np.float64)
    label = _reform_from_dict(label_chunks).astype(np.float64)

    if diff_flag:
        pred  = detrend(np.cumsum(pred),  100)
        label = detrend(np.cumsum(label), 100)
    else:
        pred  = detrend(pred,  100)
        label = detrend(label, 100)

    pred_processed  = bandpass_filter(pred,  fs, low=0.6, high=3.3)
    label_processed = bandpass_filter(label, fs, low=0.6, high=3.3)

    hr_pred  = fft_peak_hz(pred_processed,  fs, 0.6, 3.3) * 60.0
    hr_label = fft_peak_hz(label_processed, fs, 0.6, 3.3) * 60.0
    snr_db   = calculate_snr(pred_processed, hr_label, fs)

    return hr_pred, hr_label, snr_db, pred_processed

In [12]:
# Blink rate detection

def detect_blink_rate(video_path, fps=30, large_box_coef=1.5):
    """Estimate blink rate (blinks/min) using eye-strip brightness analysis.

    Steps:
      1. Detect face on frame 0 with Haar Cascade.
      2. Define eye strip = rows [20%, 50%] of face bbox height.
      3. Compute mean grayscale brightness in eye strip per frame.
      4. Invert (blinks = dips) and bandpass filter [0.1, 0.9] Hz.
      5. Count peaks with minimum separation of 1 second.
    """
    frames = read_video_frames(video_path)
    T = len(frames)

    xml_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    detector = cv2.CascadeClassifier(xml_path)
    gray0 = cv2.cvtColor(frames[0], cv2.COLOR_RGB2GRAY)
    faces = detector.detectMultiScale(gray0, scaleFactor=1.3, minNeighbors=5)
    H, W = frames[0].shape[:2]

    if len(faces) > 0:
        x, y, fw, fh = max(faces, key=lambda f: f[2])
        x  = max(0, int(x  - (large_box_coef - 1.0) / 2.0 * fw))
        y  = max(0, int(y  - (large_box_coef - 1.0) / 2.0 * fh))
        fw = min(int(fw * large_box_coef), W - x)
        fh = min(int(fh * large_box_coef), H - y)
    else:
        x, y, fw, fh = 0, 0, W, H

    eye_top    = y + int(fh * 0.20)
    eye_bottom = y + int(fh * 0.50)
    eye_left   = x
    eye_right  = x + fw

    brightness = np.array([
        np.mean(cv2.cvtColor(f, cv2.COLOR_RGB2GRAY)[eye_top:eye_bottom, eye_left:eye_right])
        for f in frames
    ])

    # blinks = dips in brightness -> invert for find_peaks
    brightness_inv = -brightness.astype(np.float64)

    # bandpass [0.1, 0.9] Hz = [6, 54] blinks/min
    b, a = signal.butter(2, [0.1 / fps * 2, 0.9 / fps * 2], btype="bandpass")
    filtered = signal.filtfilt(b, a, brightness_inv)

    # minimum 1 second between detected blinks
    min_dist = int(fps)
    peaks, _ = signal.find_peaks(filtered, distance=min_dist)

    duration_min = T / fps / 60.0
    return len(peaks) / duration_min


# Compute blink rate for each subject
blink_rate_dict = {}
for subj in subjects:
    subj_key   = subj["subj_key"]
    video_path = subj["video_path"]
    print(f"  {subj_key} ...", end=" ", flush=True)
    blink_rate = detect_blink_rate(video_path, fps=VIDEO_FPS)
    blink_rate_dict[subj_key] = blink_rate
    print(f"{blink_rate:.2f} blinks/min")

print("\nBlink rates:", {k: f"{v:.2f}" for k, v in blink_rate_dict.items()})

  S000 ... 30.85 blinks/min
  S001 ... 30.24 blinks/min
  S002 ... 29.39 blinks/min
  S003 ... 28.80 blinks/min
  S004 ... 33.65 blinks/min

Blink rates: {'S000': '30.85', 'S001': '30.24', 'S002': '29.39', 'S003': '28.80', 'S004': '33.65'}


In [13]:
# Inference loop with model-specific forward, export to results_groupF/

# lookup: subj_key -> subject info dict (contains gt_hr)
subjects_by_key = {s["subj_key"]: s for s in subjects}

# blink_rate_dict is populated by the blink detection cell;
# fall back to empty dict if that cell has not been run yet.
if "blink_rate_dict" not in dir():
    blink_rate_dict = {}


def build_model(arch, model_path):
    """Instantiate model by architecture name and load weights."""
    full_path = os.path.join(REPO_ROOT, model_path)
    state_dict = torch.load(full_path, map_location=DEVICE)

    # strip 'module.' prefix if present (DataParallel artifact)
    if any(k.startswith("module.") for k in state_dict.keys()):
        state_dict = {k[len("module."):]: v for k, v in state_dict.items()}

    if arch == "iBVPNet":
        model = iBVPNet(frames=CHUNK_LENGTH, in_channels=3)
        model.load_state_dict(state_dict)
    elif arch == "FactorizePhys":
        MD_CONFIG = {
            "FRAME_NUM": CHUNK_LENGTH,
            "MD_FSAM": True,
            "MD_TYPE": "NMF",
            "MD_TRANSFORM": "T_KAB",
            "MD_R": 1,
            "MD_S": 1,
            "MD_STEPS": 3,
            "MD_INFERENCE": True,
            "MD_RESIDUAL": True,
        }
        model = FactorizePhys(
            frames=CHUNK_LENGTH,
            md_config=MD_CONFIG,
            in_channels=3,
            dropout=0.1,
            device=torch.device(DEVICE),
        )
        model.load_state_dict(state_dict, strict=False)
    else:
        raise ValueError(f"Unknown architecture: {arch}")

    model = model.to(DEVICE)
    model.eval()
    return model


def run_inference(model, arch, loader):
    """Run inference and return per-subject prediction/label dicts."""
    preds_dict  = {}  # subj_key -> {chunk_id: np.ndarray (CHUNK_LENGTH,)}
    labels_dict = {}

    with torch.no_grad():
        for batch in tqdm(loader, desc="Inference"):
            data_t, labels_t, batch_subjects, batch_chunk_ids = batch

            # data_t shape: (N, C, T, H, W) = NCDHW
            data_in = data_t.float().to(DEVICE)

            # Both models use torch.diff internally and need T+1 frames.
            # Pad by repeating last frame in temporal dim.
            last_frame = data_in[:, :, -1:, :, :].clone()
            data_padded = torch.cat([data_in, last_frame], dim=2)  # (N, 3, T+1, H, W)

            if arch == "iBVPNet":
                pred_ppg = model(data_padded)  # (N, T)
            elif arch == "FactorizePhys":
                out = model(data_padded)
                pred_ppg = out[0]  # (N, T)
            else:
                raise ValueError(f"Unknown architecture: {arch}")

            pred_np  = pred_ppg.cpu().numpy()                    # (N, T)
            label_np = labels_t.numpy()                          # (N, T)

            N = data_t.shape[0]
            for i in range(N):
                subj = batch_subjects[i]
                cid  = int(batch_chunk_ids[i])

                if subj not in preds_dict:
                    preds_dict[subj]  = {}
                    labels_dict[subj] = {}

                preds_dict[subj][cid]  = pred_np[i]   # (T,)
                labels_dict[subj][cid] = label_np[i]  # (T,)

    return preds_dict, labels_dict


# Run all models
FS = VIDEO_FPS

for model_name, arch, model_path in MODELS:
    print(f"\n{'='*60}")
    print(f"Model: {model_name}  ({arch})")
    print(f"Weights: {model_path}")
    print(f"{'='*60}")

    model = build_model(arch, model_path)
    num_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {num_params:,}")

    preds_dict, labels_dict = run_inference(model, arch, loader)
    print(f"Subjects: {sorted(preds_dict.keys())}")

    # Per-subject results
    per_subject_results = []
    hr_preds_all = []
    gt_hrs_all   = []
    snr_all      = []

    print(f"\n{'Subject':<10} {'HR_pred':>10} {'HR_gt':>10} {'HR_err':>8} {'SNR':>7} {'Blinks':>7}")
    print("-" * 60)

    for subj_key in sorted(preds_dict.keys()):
        hr_pred, hr_label, _, pred_processed = process_bvp(
            preds_dict[subj_key], labels_dict[subj_key], fs=FS, diff_flag=False
        )

        gt_hr      = subjects_by_key[subj_key]["gt_hr"]
        snr_db     = calculate_snr(pred_processed, gt_hr, FS)
        blink_rate = blink_rate_dict.get(subj_key, float("nan"))
        hr_err     = hr_pred - gt_hr

        # map subj_key ("S000") back to original ID ("S_000")
        subj_id = subj_key[0] + "_" + subj_key[1:]

        per_subject_results.append({
            "name":                subj_id,
            "predicted_heartrate": hr_pred,
            "real_heartrate":      gt_hr,
            "heartrate_error":     hr_err,
            "blink_rate":          blink_rate,
            "snr_db":              snr_db,
        })

        hr_preds_all.append(hr_pred)
        gt_hrs_all.append(gt_hr)
        snr_all.append(snr_db)

        print(f"{subj_id:<10} {hr_pred:>10.3f} {gt_hr:>10.3f} {hr_err:>8.3f} {snr_db:>7.2f} {blink_rate:>7.2f}")

    hr_preds_all = np.array(hr_preds_all)
    gt_hrs_all   = np.array(gt_hrs_all)
    snr_all      = np.array(snr_all)

    # Aggregate metrics
    n = len(hr_preds_all)
    assert n > 0, "No subjects to evaluate."

    err   = hr_preds_all - gt_hrs_all
    abs_e = np.abs(err)
    sq_e  = err ** 2
    rel_e = abs_e / (np.abs(gt_hrs_all) + 1e-9)

    mae       = float(np.mean(abs_e))
    mae_se    = float(np.std(abs_e) / np.sqrt(n))
    rmse      = float(np.sqrt(np.mean(sq_e)))
    rmse_se   = float(np.sqrt(np.std(sq_e) / np.sqrt(n)))
    mape      = float(np.mean(rel_e) * 100.0)
    mape_se   = float(np.std(rel_e) / np.sqrt(n) * 100.0)

    if n >= 2:
        pearson_r  = float(np.corrcoef(hr_preds_all, gt_hrs_all)[0, 1])
        pearson_se = float(np.sqrt(max(0.0, (1 - pearson_r ** 2) / (n - 2))))
    else:
        pearson_r, pearson_se = float("nan"), float("nan")

    mean_snr    = float(np.mean(snr_all))
    mean_snr_se = float(np.std(snr_all) / np.sqrt(n))

    print(f"\nAggregate metrics for {model_name}:")
    print(f"  MAE     : {mae:.4f} +/- {mae_se:.4f} bpm")
    print(f"  RMSE    : {rmse:.4f} +/- {rmse_se:.4f} bpm")
    print(f"  MAPE    : {mape:.4f} +/- {mape_se:.4f} %")
    print(f"  Pearson : {pearson_r:.4f} +/- {pearson_se:.4f}")
    print(f"  SNR     : {mean_snr:.4f} +/- {mean_snr_se:.4f} dB")

    # Export results
    metrics_dict = {
        "model":      model_name,
        "architecture": arch,
        "n_subjects": n,
        "evaluation_method": "FFT",
        "label_type": LABEL_TYPE,
        "bvp_bandpass_hz":   [0.6, 3.3],
        "aggregate_metrics": {
            "MAE":     {"value": mae,      "se": mae_se,      "unit": "bpm"},
            "RMSE":    {"value": rmse,     "se": rmse_se,     "unit": "bpm"},
            "MAPE":    {"value": mape,     "se": mape_se,     "unit": "%"},
            "Pearson": {"value": pearson_r, "se": pearson_se, "unit": ""},
            "SNR":     {"value": mean_snr, "se": mean_snr_se, "unit": "dB"},
        },
        "per_subject": [
            {
                "name":               r["name"],
                "predicted_heartrate": r["predicted_heartrate"],
                "real_heartrate":      r["real_heartrate"],
                "heartrate_error":     r["heartrate_error"],
                "snr_db":              r["snr_db"],
            }
            for r in per_subject_results
        ],
    }

    model_output_dir = os.path.join(OUTPUT_DIR, model_name)
    os.makedirs(model_output_dir, exist_ok=True)

    json_path = os.path.join(model_output_dir, "metrics.json")
    with open(json_path, "w") as fh:
        json.dump(metrics_dict, fh, indent=2)
    print(f"\n  Metrics saved to: {json_path}")

    csv_rows = []
    for r in per_subject_results:
        csv_rows.append({
            "name":               r["name"],
            "predicted_heartrate": r["predicted_heartrate"],
            "real_heartrate":      r["real_heartrate"],
            "heartrate_error":     r["heartrate_error"],
            "blink_rate":          r["blink_rate"],
        })

    results_df = pd.DataFrame(csv_rows, columns=[
        "name", "predicted_heartrate", "real_heartrate",
        "heartrate_error", "blink_rate"
    ])

    csv_path = os.path.join(model_output_dir, "ppg_results.csv")
    results_df.to_csv(csv_path, index=False)
    print(f"  CSV saved to: {csv_path}")
    print()
    print(results_df.to_string(index=False))

    # Clean up model from GPU
    del model
    torch.cuda.empty_cache()

print(f"\n\nAll models complete. Results in: {OUTPUT_DIR}")


Model: PURE_iBVPNet  (iBVPNet)
Weights: final_model_release/PURE_iBVPNet.pth
Total parameters: 1,426,865


Inference: 100%|██████████| 21/21 [00:03<00:00,  6.53it/s]


Subjects: ['S000', 'S001', 'S002', 'S003', 'S004']

Subject       HR_pred      HR_gt   HR_err     SNR  Blinks
------------------------------------------------------------
S_000          76.904     80.493   -3.589   -0.66   30.85
S_001          88.330     87.971    0.359   -2.24   30.24
S_002          70.752     74.013   -3.261    1.52   29.39
S_003          65.039     65.451   -0.412    6.04   28.80
S_004          76.904     76.826    0.079    3.82   33.65

Aggregate metrics for PURE_iBVPNet:
  MAE     : 1.5399 +/- 0.6917 bpm
  RMSE    : 2.1825 +/- 1.6041 bpm
  MAPE    : 2.0009 +/- 0.8909 %
  Pearson : 0.9758 +/- 0.1263
  SNR     : 1.6970 +/- 1.3337 dB

  Metrics saved to: /home/naver/disk2/HoangDPB/rPPG-Toolbox/results/groupF/reading/PURE_iBVPNet/metrics.json
  CSV saved to: /home/naver/disk2/HoangDPB/rPPG-Toolbox/results/groupF/reading/PURE_iBVPNet/ppg_results.csv

  name  predicted_heartrate  real_heartrate  heartrate_error  blink_rate
 S_000            76.904297       80.492958    

Inference: 100%|██████████| 21/21 [00:00<00:00, 27.28it/s]


Subjects: ['S000', 'S001', 'S002', 'S003', 'S004']

Subject       HR_pred      HR_gt   HR_err     SNR  Blinks
------------------------------------------------------------
S_000          79.541     80.493   -0.952    0.24   30.85
S_001          87.012     87.971   -0.959    1.54   30.24
S_002          70.752     74.013   -3.261    2.31   29.39
S_003          65.039     65.451   -0.412    5.79   28.80
S_004          76.904     76.826    0.079    4.29   33.65

Aggregate metrics for PURE_FactorizePhys:
  MAE     : 1.1326 +/- 0.4989 bpm
  RMSE    : 1.5897 +/- 1.3492 bpm
  MAPE    : 1.4822 +/- 0.6759 %
  Pearson : 0.9883 +/- 0.0881
  SNR     : 2.8340 +/- 0.8842 dB

  Metrics saved to: /home/naver/disk2/HoangDPB/rPPG-Toolbox/results/groupF/reading/PURE_FactorizePhys/metrics.json
  CSV saved to: /home/naver/disk2/HoangDPB/rPPG-Toolbox/results/groupF/reading/PURE_FactorizePhys/ppg_results.csv

  name  predicted_heartrate  real_heartrate  heartrate_error  blink_rate
 S_000            79.541016  

...

Subjects: ['S000', 'S001', 'S002', 'S003', 'S004']

Subject       HR_pred      HR_gt   HR_err     SNR  Blinks
------------------------------------------------------------
S_000          79.541     80.493   -0.952    0.74   30.85
S_001          87.012     87.971   -0.959    2.94   30.24
S_002          72.949     74.013   -1.064    2.48   29.39
S_003          65.039     65.451   -0.412    7.69   28.80
S_004          76.904     76.826    0.079    4.95   33.65

Aggregate metrics for iBVP_FactorizePhys:
  MAE     : 0.6931 +/- 0.1711 bpm
  RMSE    : 0.7917 +/- 0.4486 bpm
  MAPE    : 0.8885 +/- 0.2110 %
  Pearson : 0.9985 +/- 0.0320
  SNR     : 3.7593 +/- 1.0640 dB

  Metrics saved to: /home/naver/disk2/HoangDPB/rPPG-Toolbox/results/groupF/reading/iBVP_FactorizePhys/metrics.json
  CSV saved to: /home/naver/disk2/HoangDPB/rPPG-Toolbox/results/groupF/reading/iBVP_FactorizePhys/ppg_results.csv

  name  predicted_heartrate  real_heartrate  heartrate_error  blink_rate
 S_000            79.541016  

Inference: 100%|██████████| 21/21 [00:00<00:00, 31.53it/s]


Subjects: ['S000', 'S001', 'S002', 'S003', 'S004']

Subject       HR_pred      HR_gt   HR_err     SNR  Blinks
------------------------------------------------------------
S_000          76.904     80.493   -3.589   -2.17   30.85
S_001          87.012     87.971   -0.959   -1.70   30.24
S_002          70.752     74.013   -3.261   -0.92   29.39
S_003          65.039     65.451   -0.412    2.08   28.80
S_004          76.904     76.826    0.079    1.11   33.65

Aggregate metrics for SCAMPS_FactorizePhys:
  MAE     : 1.6599 +/- 0.6582 bpm
  RMSE    : 2.2185 +/- 1.5873 bpm
  MAPE    : 2.1374 +/- 0.8495 %
  Pearson : 0.9792 +/- 0.1172
  SNR     : -0.3209 +/- 0.7348 dB

  Metrics saved to: /home/naver/disk2/HoangDPB/rPPG-Toolbox/results/groupF/reading/SCAMPS_FactorizePhys/metrics.json
  CSV saved to: /home/naver/disk2/HoangDPB/rPPG-Toolbox/results/groupF/reading/SCAMPS_FactorizePhys/ppg_results.csv

  name  predicted_heartrate  real_heartrate  heartrate_error  blink_rate
 S_000            76.9

Inference: 100%|██████████| 21/21 [00:00<00:00, 31.66it/s]


Subjects: ['S000', 'S001', 'S002', 'S003', 'S004']

Subject       HR_pred      HR_gt   HR_err     SNR  Blinks
------------------------------------------------------------
S_000          79.541     80.493   -0.952   -1.25   30.85
S_001          88.330     87.971    0.359    0.87   30.24
S_002          70.752     74.013   -3.261    1.94   29.39
S_003          65.039     65.451   -0.412    4.93   28.80
S_004          76.904     76.826    0.079    3.76   33.65

Aggregate metrics for UBFC-rPPG_FactorizePhys:
  MAE     : 1.0125 +/- 0.5184 bpm
  RMSE    : 1.5391 +/- 1.3614 bpm
  MAPE    : 1.3458 +/- 0.7022 %
  Pearson : 0.9879 +/- 0.0896
  SNR     : 2.0477 +/- 0.9704 dB

  Metrics saved to: /home/naver/disk2/HoangDPB/rPPG-Toolbox/results/groupF/reading/UBFC-rPPG_FactorizePhys/metrics.json
  CSV saved to: /home/naver/disk2/HoangDPB/rPPG-Toolbox/results/groupF/reading/UBFC-rPPG_FactorizePhys/ppg_results.csv

  name  predicted_heartrate  real_heartrate  heartrate_error  blink_rate
 S_000        